# Example pipeline data preparation.

## Prepare notebook.

### Flags for Autoreloading.

In [7]:
%reload_ext autoreload
%autoreload 2

### Import the libraries.

In [8]:
from sc_flow.data import DataManager
from sc_flow.data.sim import get_dummy_adata
from sc_flow.data.samplers import FTrainSampler, FValidationSampler

### Generate dummy data.

In [9]:
BIG = False
if BIG:
    n_obs_pert = 1_000_000
    n_obs_ctrl = 500_000
else:
    n_obs_pert = 10000
    n_obs_ctrl = 5000
adata = get_dummy_adata(n_obs_pert=n_obs_pert, n_obs_ctrl=n_obs_ctrl)
adata

/Users/lorenzo.consoli/micromamba/envs/sc-flow-tools/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/Users/lorenzo.consoli/micromamba/envs/sc-flow-tools/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


AnnData object with n_obs × n_vars = 15000 × 400
    obs: 'drugA', 'drugB', 'koA', 'koB', 'target', 'source_split', 'is_control'
    uns: 'drug', 'ko', 'source_split'
    obsm: 'drugA_time', 'drugA_dose', 'drugB_time', 'drugB_dose', 'koA_time', 'koA_dose', 'koB_time', 'koB_dose', 'paired_condition', 'X_repr', 'target_variable', 'X_src', 'X_tgt'

## Case 1. Loading only state data.

### Initialize data manager and get data.

In [10]:
dm = DataManager(
    sample_rep="X_repr",
)
train_collection = dm.compile_adata(adata)

train_sampler = FTrainSampler(
    train_collection,  # the tree containing the data
    lambda x: x,  # the function to process the nodes
    n_groups=1,  # the number of nodes to load
    replace_groups=True,  # whether to sample nodes with replacement
    replace_samples=True,  # whether to sample observations from nodes with replacement
    use_groups_weights=True,  # whether to weight sampling by frequency
)
val_sampler = FValidationSampler(train_collection, lambda x: x, replace_groups=True, replace_samples=True)
batch = train_sampler.sample()
batch[0], val_sampler[0]

100%|██████████| 1/1 [00:00<00:00, 7943.76it/s]


(MatchedData:
  * (target) -> 	DistributionData:
 	 * n_obs=512
 	 state=StateData(n_obs=512, spatial_dims=(200,))
 	 target=None
 	 condition=None
 	 groups=CategoricalData(n_obs=512, n_vars=0, columns=[], repr_dict_keys=[], categorical_encoders_keys=[])
 	 source_coupling=CouplingData(n_obs=512, linear(spatial_dims=(200,)), quadratic=None)
 	 target_coupling=CouplingData(n_obs=512, linear(spatial_dims=(200,)), quadratic=None),
 MatchedData:
  * (target) -> 	DistributionData:
 	 * n_obs=10000
 	 state=StateData(n_obs=10000, spatial_dims=(200,))
 	 target=None
 	 condition=None
 	 groups=CategoricalData(n_obs=10000, n_vars=0, columns=[], repr_dict_keys=[], categorical_encoders_keys=[])
 	 source_coupling=CouplingData(n_obs=10000, linear(spatial_dims=(200,)), quadratic=None)
 	 target_coupling=CouplingData(n_obs=10000, linear(spatial_dims=(200,)), quadratic=None))

## Case 2: Grouping data based on source split.

### Initialize data manager and get data.

In [11]:
dm = DataManager(
    sample_rep="X_repr",
    groups=["source_split"],
    groups_encoding={"source_split": "one-hot"},
)
train_collection = dm.compile_adata(adata)

train_sampler = FTrainSampler(train_collection, lambda x: x, n_groups=1, replace_groups=True, replace_samples=True)
val_sampler = FValidationSampler(train_collection, lambda x: x, replace_groups=True, replace_samples=True)
batch = train_sampler.sample()
batch[0], val_sampler[0]

ValueError: DistributionData is not sorted by its annotation columns. Either pass sort=True to compile_adata() or call DataManager.sort_adata(adata) before compilation.

## Case 2: Grouping data based on source split and condition.


### Initialize data manager and get data.

In [6]:
dm = DataManager(
    sample_rep="X_repr",
    conditions={
        "drug": ("drugA", "drugB"),
        "ko": ("koA", "koB"),
    },
    conditions_reps={
        "drug": "drug",
        "ko": "ko",
    },
)
train_collection = dm.compile_adata(adata)

train_sampler = FTrainSampler(train_collection, lambda x: x, n_groups=1, replace_groups=True, replace_samples=True)
val_sampler = FValidationSampler(train_collection, lambda x: x, replace_groups=True, replace_samples=True)
batch = train_sampler.sample()
batch[0], val_sampler[0]

100%|██████████| 226/226 [00:00<00:00, 3099.79it/s]


(MatchedData:
  * (target) -> 	DistributionData:
 	 * n_obs=512
 	 state=StateData(n_obs=512, spatial_dims=(200,))
 	 target=None
 	 condition=MixedTypeData(n_obs=512, categorical(n_vars=4, columns=['drugA', 'drugB', 'koA', 'koB']), continuous=None
 	 groups=CategoricalData(n_obs=512, n_vars=0, columns=[], repr_dict_keys=[], categorical_encoders_keys=[])
 	 source_coupling=CouplingData(n_obs=512, linear(spatial_dims=(200,)), quadratic=None)
 	 target_coupling=CouplingData(n_obs=512, linear(spatial_dims=(200,)), quadratic=None),
 MatchedData:
  * (target) -> 	DistributionData:
 	 * n_obs=10000
 	 state=StateData(n_obs=10000, spatial_dims=(200,))
 	 target=None
 	 condition=MixedTypeData(n_obs=10000, categorical(n_vars=4, columns=['drugA', 'drugB', 'koA', 'koB']), continuous=None
 	 groups=CategoricalData(n_obs=10000, n_vars=0, columns=[], repr_dict_keys=[], categorical_encoders_keys=[])
 	 source_coupling=CouplingData(n_obs=10000, linear(spatial_dims=(200,)), quadratic=None)
 	 target_c

## Case 3: Grouping data based on source split and condition with controls.


### Initialize data manager and get data.

In [7]:
dm = DataManager(
    sample_rep="X_tgt",
    conditions={
        "drug": ("drugA", "drugB"),
        "ko": ("koA", "koB"),
    },
    conditions_reps={
        "drug": "drug",
        "ko": "ko",
    },
    conditions_covariates=["paired_condition"],
    groups=["source_split"],
    groups_encoding={"source_split": "one-hot"},
    control_values_dict={"drug": "control", "ko": "control"},
    source_rep="X_src",
    n_shared_dims=10,
)
train_collection = dm.compile_adata(adata)

train_sampler = FTrainSampler(train_collection, lambda x: x, n_groups=1, replace_groups=True, replace_samples=True)
val_sampler = FValidationSampler(train_collection, lambda x: x, replace_groups=True, replace_samples=True)
batch = train_sampler.sample()
batch[0], val_sampler[0]

100%|██████████| 225/225 [00:00<00:00, 1605.95it/s]


(MatchedData:
  * (target) -> 	DistributionData:
 	 * n_obs=512
 	 state=StateData(n_obs=512, spatial_dims=(22,))
 	 target=None
 	 condition=MixedTypeData(n_obs=512, categorical(n_vars=4, columns=['drugA', 'drugB', 'koA', 'koB']), continuous(keys=['paired_condition'], spatial_dims={'paired_condition': 100})
 	 groups=CategoricalData(n_obs=512, n_vars=1, columns=['source_split'], repr_dict_keys=[], categorical_encoders_keys=['source_split'])
 	 source_coupling=CouplingData(n_obs=512, linear(spatial_dims=(10,)), quadratic(spatial_dims=(6,)))
 	 target_coupling=CouplingData(n_obs=512, linear(spatial_dims=(10,)), quadratic(spatial_dims=(12,)))
  * (source) -> 	DistributionData:
 	 * n_obs=512
 	 state=StateData(n_obs=512, spatial_dims=(22,))
 	 target=None
 	 condition=MixedTypeData(n_obs=512, categorical(n_vars=4, columns=['drugA', 'drugB', 'koA', 'koB']), continuous(keys=['paired_condition'], spatial_dims={'paired_condition': 100})
 	 groups=CategoricalData(n_obs=512, n_vars=1, columns=

### Sampling a bunch of times.

In [8]:
from tqdm import tqdm

for _ in tqdm(range(1000)):
    batch = train_sampler.sample()

100%|██████████| 1000/1000 [00:00<00:00, 2369.44it/s]
